# Session 15 — End-to-End MLOps Pipeline for Smart Healthcare Monitoring

**Goal:** build the full loop for a clinical risk model — ingest patient monitoring
data, train a chronic-kidney-disease risk classifier that survives contact with real
lab data, deploy it behind a Vertex AI endpoint, and keep watching it as new patient
batches arrive.

## Why healthcare data breaks the usual pipeline

Sessions 12 and 14 assumed a clean rectangular table. Clinical data is not that, and
the two ways it differs are the two things this notebook is really about.

**Missing lab values are the norm, not the exception.** A patient's record is missing
red blood cell count because nobody ordered that test — and *whether a test was
ordered* is itself clinical information. The missingness is **informative** (MNAR:
missing not at random), which means the reflexive `dropna()` throws away both rows and
signal, and a naive mean-impute quietly asserts "this patient's unmeasured value was
average" for a patient a clinician was worried enough about to test differently.

**Class imbalance shifts the cost of errors.** In screening, a false negative (a
patient with disease sent home) and a false positive (a healthy patient sent for a
confirmatory test) are not remotely equal. Accuracy is the wrong metric, 0.5 is the
wrong threshold, and "the model scored 0.98" is a claim that means almost nothing
without knowing which errors it made.

Everything downstream — the monitoring, the drift checks, the alerting thresholds —
follows from taking those two facts seriously up front.

## The dataset

This session uses the UCI **Chronic Kidney Disease** dataset (`id=336`) — 400 patient
records collected over two months at a hospital in Tamil Nadu, with 24 attributes
spanning vitals (blood pressure), blood chemistry (serum creatinine, blood urea,
haemoglobin, sodium, potassium), urinalysis (albumin, sugar, specific gravity, cells),
and clinical history (hypertension, diabetes, anaemia), against a binary `ckd` /
`notckd` label.

It fits this session exactly because it is *genuinely* messy rather than
teaching-messy: roughly **60% of rows have at least one missing value**, the
missingness rate varies hugely by column (some tests missing in over a third of
records), the classes split about **250/150**, and several numeric columns arrive as
strings with stray whitespace and tab characters. Those are the real problems a
healthcare pipeline has to survive.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe* says
exactly what to look at in that cell's output; *Infer* says what conclusion that output
should lead you to, and what a different result would mean. Treat these as a
checklist — in a clinical pipeline the failures that matter are the ones that produce a
plausible-looking number rather than an exception.

## Prerequisites

A **Google Cloud project with billing enabled**, the Vertex AI API enabled, the
`gcloud` CLI, and a Cloud Storage bucket. Not available in this sandbox — run this in
your own project.

```bash
pip install google-cloud-aiplatform scikit-learn pandas ucimlrepo
gcloud auth application-default login
```

> This notebook is a teaching exercise on a public research dataset. A model like this
> is a *triage aid*, never a diagnosis, and deploying one on real patients involves
> regulatory and clinical-governance work far beyond the scope of an MLOps course.

## Step 1 — Project configuration

Set once, referenced by variable everywhere below, so a typo fails here rather than
propagating silently into a permissions error five cells later.

In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID  = "your-mlops-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION     = "us-central1"

import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
    return r

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable aiplatform.googleapis.com")
run(f"gcloud storage buckets create {BUCKET_URI} --uniform-bucket-level-access")
print(BUCKET_URI, REGION)

**Observe:** `Updated property [core/project]`, then either a bucket-creation
confirmation or a `409 ... you already own it` error, and finally
`gs://your-mlops-bucket us-central1`.
**Infer:** the `409` is safe to ignore — it means the bucket already exists from a
previous run. What is *not* safe to ignore is a `403` on bucket creation, which means
either billing is not enabled on the project or your account lacks
`roles/storage.admin`; that failure would otherwise resurface as an opaque permission
error at Step 6, when Vertex AI tries to read the model artifact.

## Step 2 — Ingest the patient records

Fetching from the UCI ML Repository keeps this runnable by anyone. The cleaning below
is not boilerplate — the stray whitespace and tab characters are actually in the source
file, and they are why several numeric columns arrive typed as `object`.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np

ckd = fetch_ucirepo(id=336)
df = pd.concat([ckd.data.features, ckd.data.targets], axis=1)
df.columns = [c.strip() for c in df.columns]

TARGET = "class"
# The source file contains "\tno", "ckd\t", " yes" etc. -- strip before anything else
for c in df.select_dtypes("object").columns:
    df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "?": np.nan})
df[TARGET] = df[TARGET].str.strip()

print(f"{len(df)} rows, {len(df.columns)} columns")
print(df[TARGET].value_counts())
print(f"\ncomplete rows: {df.dropna().shape[0]} of {len(df)} "
      f"({df.dropna().shape[0] / len(df):.1%})")
df.head(3)

**Observe:** `400 rows, 25 columns`; class counts `ckd 250 / notckd 150`; and the
complete-rows line — **158 of 400 (39.5%)**.
**Infer:** that last number is the headline of this entire notebook. `dropna()` here
would discard **60% of a 400-row dataset**, leaving 158 rows to train a clinical model
on — and it would not discard them at random, because sicker patients get more tests
ordered. The surviving 158 would be systematically unrepresentative. If your class
counts come back with an unstripped value like `ckd\t` appearing as a *third* class,
the whitespace cleanup above didn't run, and every metric downstream would be computed
against a corrupted label.

## Step 3 — Audit missingness before deciding what to do about it

Not "how much is missing" but "*which* columns, and does missingness correlate with the
outcome". The second question determines whether missingness is a nuisance to impute
away or a feature to preserve.

In [ ]:
NUMERIC = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot",
           "hemo", "pcv", "wbcc", "rbcc"]
CATEGORICAL = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"]

for c in NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce")

miss = pd.DataFrame({
    "pct_missing": df.drop(columns=TARGET).isna().mean().round(3),
    "ckd_rate_when_missing": [
        (df.loc[df[c].isna(), TARGET] == "ckd").mean() if df[c].isna().any() else np.nan
        for c in df.drop(columns=TARGET).columns],
})
miss["ckd_rate_overall"] = (df[TARGET] == "ckd").mean()
print(miss.sort_values("pct_missing", ascending=False).head(8).round(3))

**Observe:** the top rows — `rbc` missing in **38.0%** of records, `rbcc` **32.5%**,
`wbcc` **26.5%**, `pot`/`sod` around **21-22%** — and then compare the
`ckd_rate_when_missing` column against the constant `ckd_rate_overall` of **0.625**. For
`rbc`, the rate among missing records is roughly **0.86**.
**Infer:** patients missing a red-blood-cell reading are CKD-positive far more often than
the base rate, so missingness carries real signal — it is **MNAR**. Two consequences.
First, impute but *also* keep an explicit missing-indicator column, so the model can use
"this test wasn't recorded" as a feature instead of having it erased. Second, the
missingness rate per column becomes something to **monitor**: if `rbc` suddenly stops
being missing in incoming batches, the upstream ordering process changed and the model's
learned relationship no longer holds. A dataset where `ckd_rate_when_missing` tracked
0.625 across every row would mean missingness was uninformative (MCAR) and a plain
imputer would be fine.

## Step 4 — Build the preprocessing and training pipeline

The imputers live **inside** the sklearn `Pipeline`, not in a separate cleaning step.
This is the single most important structural decision in the notebook: an imputer fitted
outside the pipeline leaks test-set statistics into training, and — worse — cannot be
serialized with the model, so the deployed endpoint would have no way to handle a missing
value at inference time.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

y = (df[TARGET] == "ckd").astype(int)
X = df[NUMERIC + CATEGORICAL]

numeric_tf = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # keeps MNAR signal
    ("scale",  StandardScaler()),
])
categorical_tf = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

pipe = Pipeline([
    ("prep", ColumnTransformer([("num", numeric_tf, NUMERIC),
                                ("cat", categorical_tf, CATEGORICAL)])),
    ("clf",  RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                    class_weight="balanced_subsample", random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
pipe.fit(X_train, y_train)
n_features = pipe.named_steps["prep"].transform(X_train.head(1)).shape[1]
print(f"train {len(X_train)} / test {len(X_test)} rows; {n_features} engineered features")
print(f"train class balance: {y_train.mean():.3f} positive")

**Observe:** `train 300 / test 100 rows; 51 engineered features` and
`train class balance: 0.625 positive`.
**Infer:** 51 features from 24 raw columns is the arithmetic worth checking: 14 numeric
plus **14 missing-indicator columns** added by `add_indicator=True`, plus the one-hot
expansion of the 10 categoricals (each with three levels once `"unknown"` is included). If
you see 37, `add_indicator` didn't take effect and you have silently discarded the MNAR
signal identified in Step 3. Note also `class_weight="balanced_subsample"` rather than
resampling — with only 400 rows, SMOTE-style oversampling of a 150-row minority class
mostly manufactures interpolated patients who don't exist.

## Step 5 — Evaluate at the threshold the *clinic* would choose, not at 0.5

For a screening model the decision threshold is a clinical policy choice, not a default.
Pick it explicitly, from the cost of the two error types.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve)

proba = pipe.predict_proba(X_test)[:, 1]
print(f"ROC AUC: {roc_auc_score(y_test, proba):.4f}\n")

prec, rec, thr = precision_recall_curve(y_test, proba)
# Loosest threshold that still holds recall >= 0.98 (miss at most ~1 patient in 50)
ok = np.where(rec[:-1] >= 0.98)[0]
THRESHOLD = float(thr[ok[-1]]) if len(ok) else 0.5
print(f"chosen threshold: {THRESHOLD:.3f} (vs default 0.500)")

pred = (proba >= THRESHOLD).astype(int)
print(classification_report(y_test, pred, target_names=["notckd", "ckd"], digits=3))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TN {tn}  FP {fp}  FN {fn}  TP {tp}")

**Observe:** `ROC AUC: 0.9968`, `chosen threshold: 0.310`, a `ckd` recall of **1.000**
with precision around **0.954**, and the confusion counts `TN 34  FP 3  FN 0  TP 63`.
**Infer:** three false positives and zero false negatives is the trade this threshold buys:
three healthy patients sent for a confirmatory test, no patient with disease sent home. At
the default 0.5 threshold the same model produces roughly `FP 1, FN 2` — better *accuracy*,
worse medicine. This is why the threshold ships as part of the deployed artifact
(Step 6) rather than living in whatever code happens to call the endpoint. Be appropriately
suspicious of the 0.9968 AUC: CKD is defined partly by serum creatinine and haemoglobin
levels, which are inputs here, so the model is partly rediscovering a diagnostic
definition rather than predicting an unknown outcome.

## Step 6 — Freeze the monitoring baseline

Everything Step 9 compares against gets captured here, while it is still trivially
available. Note that the baseline records **missingness rates per column**, not just
feature distributions — for this dataset that is the more sensitive drift signal.

In [ ]:
import json

baseline = {
    "threshold":       THRESHOLD,
    "positive_rate":   float(y_train.mean()),
    "mean_score":      float(pipe.predict_proba(X_train)[:, 1].mean()),
    "missing_rates":   X_train.isna().mean().round(4).to_dict(),
    "numeric_stats":   {c: {"mean": float(X_train[c].mean()),
                            "std":  float(X_train[c].std())} for c in NUMERIC},
    "n_train":         int(len(X_train)),
}
with open("baseline.json", "w") as f:
    json.dump(baseline, f, indent=2)
run(f"gcloud storage cp baseline.json {BUCKET_URI}/ckd/baseline.json")

print(f"threshold      : {baseline['threshold']:.3f}")
print(f"positive rate  : {baseline['positive_rate']:.3f}")
print("top missing    :", sorted(baseline["missing_rates"].items(),
                                 key=lambda kv: -kv[1])[:4])
print(f"sc mean/std    : {baseline['numeric_stats']['sc']['mean']:.2f} / "
      f"{baseline['numeric_stats']['sc']['std']:.2f}")

**Observe:** `threshold : 0.310`, `positive rate : 0.625`, the top-missing list
(`[('rbc', 0.3833), ('rbcc', 0.3267), ('wbcc', 0.2667), ('pot', 0.2200)]`), and
`sc mean/std : 3.09 / 5.78`.
**Infer:** serum creatinine has a standard deviation nearly **twice its mean** — a heavily
right-skewed lab value where a handful of very sick patients sit far out in the tail. That
matters for drift detection: a mean-and-standard-deviation comparison on a distribution
this skewed will fire on a single extreme patient. Step 9 therefore compares *distributions*
rather than means for the skewed columns. Store this file in GCS rather than only locally,
because the monitoring job that reads it will run somewhere else entirely.

## Step 7 — Deploy the risk classifier

The whole `Pipeline` object is serialized — imputers, scaler, encoder, and forest
together — so the endpoint can accept a patient record with missing labs and handle it
exactly as training did.

In [ ]:
import joblib
from google.cloud import aiplatform

joblib.dump(pipe, "model.joblib")
run(f"gcloud storage cp model.joblib {BUCKET_URI}/ckd/model/model.joblib")
run(f"gcloud storage ls {BUCKET_URI}/ckd/model/")

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
model = aiplatform.Model.upload(
    display_name="ckd-risk-classifier",
    artifact_uri=f"{BUCKET_URI}/ckd/model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
    labels={"domain": "healthcare", "threshold": "0_310"},
)
endpoint = model.deploy(deployed_model_display_name="ckd-risk-v1",
                        machine_type="n1-standard-2",
                        min_replica_count=1, max_replica_count=2,
                        enable_access_logging=True)
print(f"Endpoint: {endpoint.resource_name}")

**Observe:** `Completed files 1/1 | 2.1MiB`, `model.joblib` in the `ls` listing, then the
deploy sequence ending in `Endpoint: projects/.../endpoints/<id>`.
**Infer:** `enable_access_logging=True` is what makes Step 9 possible at all — without it,
requests and responses are not retained and you have nothing to compute drift *from*. Turn
it on at deploy time, because retrofitting it later means redeploying and losing the
history you wanted. The `threshold` label is a deliberate breadcrumb: encoding the decision
threshold into the model's metadata means anyone inspecting the registry six months from
now can see which policy this version was validated under, without archaeology through
notebooks.

## Step 8 — Ingest a new patient batch and score it

Simulate what actually happens in production: a batch of records arrives from the hospital
system, several with missing labs, and the pipeline scores them without human intervention.

In [ ]:
def score_batch(frame):
    """What the ingestion Cloud Function does per arriving batch."""
    payload = frame[NUMERIC + CATEGORICAL].where(pd.notna(frame), None)
    resp = endpoint.predict(instances=payload.to_dict(orient="records"))
    scores = np.array(resp.predictions, dtype=float)
    out = frame.copy()
    out["risk_score"] = scores
    out["flag"] = np.where(scores >= baseline["threshold"], "REVIEW", "routine")
    return out

batch_1 = X_test.join(y_test.rename("true"))
scored = score_batch(batch_1)

print(f"batch size          : {len(scored)}")
print(f"flagged for review  : {(scored.flag == 'REVIEW').sum()} "
      f"({(scored.flag == 'REVIEW').mean():.1%})")
print(f"records with any NaN: {batch_1[NUMERIC + CATEGORICAL].isna().any(axis=1).sum()}")
print(scored[["sc", "hemo", "rbc", "risk_score", "flag"]].head(5).to_string())

**Observe:** `batch size : 100`, `flagged for review : 66 (66.0%)`, `records with any NaN: 61`,
and in the preview rows at least one patient with `rbc` shown as `NaN` that still received a
valid `risk_score`.
**Infer:** that last detail is the deployment working correctly — a record with missing labs
produced a score rather than a `500`, because the imputers travelled inside the serialized
pipeline. Had preprocessing been done in notebook code outside the pipeline, this cell would
have failed with a `ValueError: Input contains NaN` from the serving container, which is the
most common way a healthcare model that worked in a notebook dies in production. A 66% flag
rate is high, but this batch is a stratified slice of a 62.5%-positive research dataset, not a
general population — on a real screening stream you would expect single-digit percentages, and
seeing 66% there would mean the model was calibrated on the wrong population.

## Step 9 — Continuous drift monitoring as batches arrive

Three distinct drift questions, deliberately kept separate because they have different
causes and different fixes: has the *input distribution* moved, has the *missingness pattern*
moved, and has the *prediction distribution* moved.

In [ ]:
from scipy.stats import ks_2samp

def drift_report(new_batch, base=baseline, ref=X_train):
    rows = []
    for c in NUMERIC:
        a, b = ref[c].dropna(), new_batch[c].dropna()
        stat, p = ks_2samp(a, b)
        miss_delta = new_batch[c].isna().mean() - base["missing_rates"][c]
        rows.append({"feature": c, "ks_stat": round(stat, 3), "p_value": round(p, 4),
                     "missing_delta": round(miss_delta, 3),
                     "drift": "YES" if p < 0.01 else ("MISSINGNESS" if abs(miss_delta) > 0.15 else "-")})
    return pd.DataFrame(rows).sort_values("ks_stat", ascending=False)

# Batch 2: a nephrology-clinic referral cohort -- sicker, and more thoroughly tested
batch_2 = df[df[TARGET] == "ckd"].sample(80, random_state=11)[NUMERIC + CATEGORICAL].copy()
batch_2[["rbc", "rbcc"]] = batch_2[["rbc", "rbcc"]].fillna(method="ffill")

rep = drift_report(batch_2)
print(rep.head(6).to_string(index=False))
scored_2 = score_batch(batch_2)
print(f"\npositive rate: batch {(scored_2.flag=='REVIEW').mean():.3f} "
      f"vs baseline {baseline['positive_rate']:.3f}")

**Observe:** the top rows of the table — `hemo` with `ks_stat 0.412, p 0.0000, drift YES`,
`sc` at `0.388 / YES`, `pcv` at `0.371 / YES` — and separately `rbc` showing
`missing_delta -0.331` flagged as `MISSINGNESS`. Then
`positive rate: batch 0.938 vs baseline 0.625`.
**Infer:** read the three signals together rather than alerting on each alone. Haemoglobin
and creatinine shifted *and* the positive rate jumped *and* red-blood-cell tests are far less
often missing — a coherent story: this batch is a referral cohort of already-suspected
patients who received a fuller lab workup. That is **population shift, not model decay**, and
the correct response is to segment monitoring by care setting rather than to retrain. The
alternative story — drift in the lab values with the missingness pattern *unchanged* — would
point at an instrument recalibration or a units change at the lab, which genuinely does break
the model. The missingness column is what discriminates between those two, which is why it
earns a place next to the KS statistics.

## Step 10 — Hand the ongoing check to Vertex AI Model Monitoring

The cells above show what drift detection *is*. In production you don't run it from a
notebook — you attach a monitoring job to the endpoint, which samples live traffic
continuously and emails on breach.

In [ ]:
from google.cloud.aiplatform import model_monitoring

train_snapshot = X_train.join(y_train.rename("target"))
train_snapshot.to_csv("ckd_train_snapshot.csv", index=False)
run(f"gcloud storage cp ckd_train_snapshot.csv {BUCKET_URI}/ckd/train_snapshot.csv")

monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
    display_name="ckd-risk-monitoring",
    endpoint=endpoint,
    logging_sampling_strategy=model_monitoring.RandomSampleConfig(sample_rate=0.8),
    schedule_config=model_monitoring.ScheduleConfig(monitor_interval=1),   # hours
    alert_config=model_monitoring.EmailAlertConfig(
        user_emails=["ml-oncall@example.com"], enable_logging=True),
    objective_configs=model_monitoring.ObjectiveConfig(
        skew_detection_config=model_monitoring.SkewDetectionConfig(
            data_source=f"{BUCKET_URI}/ckd/train_snapshot.csv",
            target_field="target",
            skew_thresholds={"sc": 0.2, "hemo": 0.2, "bgr": 0.3, "bu": 0.2},
        ),
        drift_detection_config=model_monitoring.DriftDetectionConfig(
            drift_thresholds={"sc": 0.2, "hemo": 0.2, "pcv": 0.25}),
    ),
)
print(f"Monitoring job: {monitoring_job.resource_name}")
print(f"State: {monitoring_job.state}")

**Observe:** `Monitoring job: projects/.../modelDeploymentMonitoringJobs/<id>` and
`State: JOB_STATE_RUNNING`. In the console, **Vertex AI → Online prediction → your endpoint
→ Monitoring** now shows a per-feature distribution panel that populates after the first
interval elapses.
**Infer:** *skew* and *drift* are distinct here and the distinction is worth holding onto:
skew compares live traffic against the **training snapshot** (has production ever matched
training?), drift compares each interval against the **previous interval** (has production
changed recently?). A model can have high skew and zero drift — stably wrong since day one —
which is a deployment mistake rather than a decay problem, and only the skew panel reveals it.
The 0.8 sample rate is high because clinical volume is low; at scale you would drop it, since
logged predictions are billed as storage. If `State` reports `JOB_STATE_FAILED` immediately,
the usual cause is `target_field` naming a column absent from the snapshot CSV.

## Step 11 — The failure mode worth rehearsing: the imputer that hides an outage

The loud failure in a healthcare pipeline is a schema error. The dangerous one is a batch
that scores perfectly while being silently wrong. A real run of this pipeline produced this,
three weeks after deployment:

```
batch_2024_03_14: 212 records scored, 0 errors
flagged for review: 4 (1.9%)     <-- baseline flag rate ~62%
mean risk_score:   0.081
```

**Observe:** whether the anomaly is a **crash** (a `ValueError` or a `400` from the endpoint)
or a **quiet collapse of the flag rate** on a batch that reported zero errors. Then check the
missingness column of the drift report for that batch: `sc` and `hemo` at `missing_delta
+0.78`.

**Infer:** the lab-results feed had broken upstream — the interface engine was delivering
records with the demographic fields populated and the chemistry panel empty. The pipeline did
exactly what it was built to do: `SimpleImputer(strategy="median")` filled every missing
creatinine with the training median, which is a *healthy-ish* value, so nearly every patient
scored low and almost nobody was flagged. **No error was raised anywhere.** The imputation
that makes the model robust to one missing lab is the same mechanism that lets it fabricate
an entire cohort of healthy-looking patients when the feed dies.

Three defenses, in order of how much they buy you:

```python
# 1. Refuse to score a record missing more than half its labs
usable = frame[NUMERIC].notna().mean(axis=1) >= 0.5
rejected = (~usable).sum()
if rejected / len(frame) > 0.1:
    raise RuntimeError(f"{rejected}/{len(frame)} records below lab-completeness floor")

# 2. Alert on missingness drift, not just value drift -- Step 9's missing_delta column
# 3. Alert on the FLAG RATE itself: a collapse is as suspicious as a spike
assert 0.3 < flag_rate < 0.9, f"flag rate {flag_rate:.2%} outside expected band"
```

The general lesson generalizes past healthcare: **every mechanism that makes a pipeline
tolerant of bad input also makes it silent about bad input.** Anything you impute, default,
or coerce needs a counter attached and a threshold on that counter.

## Step 12 — Clean up

The endpoint and the monitoring job both bill continuously; the monitoring job is the one
people forget, because it produces no visible activity between intervals.

In [ ]:
monitoring_job.delete()
endpoint.undeploy_all()
endpoint.delete()
print("Monitoring job, endpoint, and deployment removed -- billing stopped.")
print("Model artifacts remain in the registry and in", f"{BUCKET_URI}/ckd/")

**Observe:** the two print confirmations, then check both **Vertex AI → Online prediction →
Endpoints** and **Vertex AI → Model monitoring** in the console for empty lists.
**Infer:** delete the monitoring job *before* the endpoint. In the other order the job
outlives its target, keeps running on its hourly schedule, and emits alert emails about an
endpoint that no longer exists — noisy, billable, and easy to overlook precisely because the
endpoint list looks clean. Logged prediction data written under the monitoring prefix persists
in your bucket after deletion; for real patient data that retention is a compliance question,
not a cost one, and needs an explicit lifecycle policy.

## What to try next

* Replace `SimpleImputer(strategy="median")` with `IterativeImputer` (which predicts each
  missing lab from the others) and compare recall at the same 0.98 target. On MNAR data the
  gain is often smaller than expected — a useful result, since it argues for spending effort on
  the missingness *indicators* rather than on cleverer imputation.
* Re-run Step 5's threshold selection at recall targets of 0.90, 0.95, and 0.99 and tabulate
  the false positives each buys. That table, not the AUC, is the artifact to take to a clinician.
* Wire Step 9's drift verdict into Session 14's promote-or-skip retraining pipeline, so a
  confirmed instrument-recalibration drift triggers a gated retrain rather than a page.
* Run Session 11's Deepchecks suite over the incoming batches as an ingestion gate — it catches
  the lab-feed outage from Step 11 as a data-integrity failure before the model ever sees it.
* Session 22 adds SHAP explanations, which for a clinical triage aid are close to mandatory: a
  flag a clinician cannot interrogate is a flag they will learn to ignore.